In [2]:
import pandas as pd
import numpy as np
import scipy.sparse as sp

In [4]:
# Load datasets
movies = pd.read_csv(r'..\Data\raw\movies.csv')
ratings = pd.read_csv(r'..\Data\raw\ratings.csv')


movies.head()
ratings.head()



,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
# merge the two datasets 
movie_ratings = pd.merge(ratings, movies,on='movieId')
movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(movie_ratings, test_size=0.2, random_state=42)

print("Training ratings:", len(train_df))
print("Test ratings:", len(test_df))

Training ratings: 80668
Test ratings: 20168


In [ ]:
from scipy.sparse import csr_matrix

user_cat = train_df['userId'].astype('category') # treats ids as a label .
movie_cat = train_df['movieId'].astype('category')

matrix = csr_matrix(

    (train_df['rating'], (movie_cat.cat.codes, user_cat.cat.codes)) # gives each label a position no.

)

print("Table shape (movies x users):", matrix.shape)
print("Ratings stored:", matrix.nnz)

Table shape (movies x users): (8983, 610)
Ratings stored: 80668


###### The idea :

### Each movie is now a row of 610 numbers (one per user). To find movies like Toy Story, the model compares Toy Story’s row with every other row and keeps the closest ones which uses cosine similarity. It asks whether two rows point in the same direction, meaning the same people liked both and disliked both. The model reports a distance (0 means identical direction, 1 means unrelated), and we flip it into a similarity (1 minus distance) so that higher means more alike.

In [ ]:
# similar rows.

from sklearn.neighbors import NearestNeighbors

# the tool: finds which rows are closest to a given row
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(matrix)
movie_ids = list(movie_cat.cat.categories)
row_of = {movie_id: row for row, movie_id in enumerate(movie_ids)}

print("Rows in the tool:", len(movie_ids))

Rows in the tool: 8983


In [16]:

movie_info = (movie_ratings[['movieId', 'title', 'genres']]
              .drop_duplicates('movieId')
              .set_index('movieId'))

movie_info.head(3)

,title,genres
movieId,,
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
6,Heat (1995),Action|Crime|Thriller


In [17]:
found = movie_info[movie_info.title.str.contains("Toy Story", regex=False)]
print(found)

                      title                                            genres
movieId                                                                      
1          Toy Story (1995)       Adventure|Animation|Children|Comedy|Fantasy
3114     Toy Story 2 (1999)       Adventure|Animation|Children|Comedy|Fantasy
78499    Toy Story 3 (2010)  Adventure|Animation|Children|Comedy|Fantasy|IMAX


In [ ]:
# find similar movies (10 closest) )

#  which movie, and which row is it?
movie_id = found.index[0]
row = row_of[movie_id]

#  ask the tool for the 11 closest rows (the first is Toy Story itself)
distances, rows = knn.kneighbors(matrix[row], n_neighbors=11)

#  turn the row numbers back into movie IDs, skipping the first (itself)
neighbor_ids = [movie_ids[r] for r in rows[0][1:]]

#  attach titles, and flip distance into similarity (higher = more alike)
result = movie_info.loc[neighbor_ids, ['title', 'genres']].copy()
result['similarity'] = (1 - distances[0][1:]).round(3)

result

,title,genres,similarity
movieId,,,
1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.469
364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.463
318,"Shawshank Redemption, The (1994)",Crime|Drama,0.460
1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.458
356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.457
260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.456
3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.453
1073,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,0.451
595,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,0.450


In [22]:
def similar_movies(title, n=10):
    # find movies whose title contains your text, and that the tool knows about
    found = movie_info[movie_info.title.str.contains(title, case=False, regex=False)
                       & movie_info.index.isin(row_of)]
    if found.empty:
        print("No movie found for:", title)
        return

    movie_id = found.index[0]
    print("Movies similar to:", found.title.iloc[0])

    # ask for n + 1, because the closest row is the movie itself
    distances, rows = knn.kneighbors(matrix[row_of[movie_id]], n_neighbors=n + 1)

    neighbor_ids = [movie_ids[r] for r in rows[0][1:]]
    result = movie_info.loc[neighbor_ids, ['title', 'genres']].copy()
    result['similarity'] = (1 - distances[0][1:]).round(3)
    return result.reset_index(drop=True)

similar_movies("Toy Story")

Movies similar to: Toy Story (1995)


,title,genres,similarity
0,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.469
1,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.463
2,"Shawshank Redemption, The (1994)",Crime|Drama,0.460
3,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.458
4,Forrest Gump (1994),Comedy|Drama|Romance|War,0.457
5,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.456
6,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.453
7,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,0.451
8,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,0.450
9,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.450


In [23]:
def similar_movies(title, n=10):
    # find movies whose title contains your text, and that the tool knows about
    found = movie_info[movie_info.title.str.contains(title, case=False, regex=False)
                       & movie_info.index.isin(row_of)]
    if found.empty:
        print("No movie found for:", title)
        return

    movie_id = found.index[0]
    print("Movies similar to:", found.title.iloc[0])

    # ask for n + 1, because the closest row is the movie itself
    distances, rows = knn.kneighbors(matrix[row_of[movie_id]], n_neighbors=n + 1)

    neighbor_ids = [movie_ids[r] for r in rows[0][1:]]
    result = movie_info.loc[neighbor_ids, ['title', 'genres']].copy()
    result['similarity'] = (1 - distances[0][1:]).round(3)
    return result.reset_index(drop=True)

similar_movies("Dumb and Dumber")


Movies similar to: Dumb & Dumber (Dumb and Dumber) (1994)


,title,genres,similarity
0,True Lies (1994),Action|Adventure|Comedy|Romance|Thriller,0.492
1,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,0.486
2,Ace Ventura: Pet Detective (1994),Comedy,0.485
3,Happy Gilmore (1996),Comedy,0.469
4,Batman (1989),Action|Crime|Thriller,0.468
5,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.464
6,Batman Forever (1995),Action|Adventure|Comedy|Crime,0.446
7,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,0.445
8,Tommy Boy (1995),Comedy,0.444
9,Stargate (1994),Action|Adventure|Sci-Fi,0.435


In [24]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

global_mean = train_df.rating.mean()
user_mean = train_df.groupby('userId').rating.mean()

# for each user: the matrix rows of the movies they rated, and the ratings they gave
user_data = {
    u: (g['row'].values, g['rating'].values)
    for u, g in train_df.assign(row=train_df.movieId.map(row_of)).groupby('userId')
}

In [25]:
# the rating guesser.
def predict_rating(user_id, movie_id, k=20):
    fallback = user_mean.get(user_id, global_mean)

    # movie never seen in practice data: nothing to compare with
    if movie_id not in row_of:
        return fallback

    rows, ratings_given = user_data[user_id]

    # how similar is this movie to each movie the user has rated?
    sims = cosine_similarity(matrix[row_of[movie_id]], matrix[rows]).ravel()

    # keep the k most similar ones the user has rated
    top = np.argsort(sims)[-k:]
    weights = sims[top]

    if weights.sum() == 0:
        return fallback

    # weighted average of the user's own ratings on those movies
    return np.dot(weights, ratings_given[top]) / weights.sum()

In [26]:
preds = [predict_rating(u, m) for u, m in zip(test_df.userId, test_df.movieId)]

rmse_knn = np.sqrt(np.mean((test_df.rating.values - np.array(preds)) ** 2))
print("Model B (item KNN) RMSE:", round(rmse_knn, 4))

Model B (item KNN) RMSE: 0.8661


In [27]:
import os, joblib
from scipy.sparse import csr_matrix, save_npz

os.makedirs("../models", exist_ok=True)

# rebuild the movie x user table from ALL ratings
all_user = movie_ratings['userId'].astype('category')
all_movie = movie_ratings['movieId'].astype('category')

full_matrix = csr_matrix(
    (movie_ratings['rating'], (all_movie.cat.codes, all_user.cat.codes))
)
full_movie_ids = list(all_movie.cat.categories)

print("Full table shape:", full_matrix.shape)
print("Ratings stored:", full_matrix.nnz)

# save the table and the row -> movie ID list
save_npz("../models/matrix.npz", full_matrix)
joblib.dump(full_movie_ids, "../models/movie_ids.pkl")

for f in ["matrix.npz", "movie_ids.pkl"]:
    print(f, round(os.path.getsize(f"../models/{f}") / 1e6, 2), "MB")

Full table shape: (9724, 610)
Ratings stored: 100836
matrix.npz 0.19 MB
movie_ids.pkl 0.03 MB
